# III. Comparaison des triptyques

Comparaison par identifiants globaux de tokens "normalisés" (`AlignID_*`).


In [6]:
import pandas as pd

TRIPTYQUES_FOLDER = "../results/csv_triptyques/"

auto = pd.read_csv(f"{TRIPTYQUES_FOLDER}triptyques_tdm_chap.csv")
manuel = pd.read_csv(f"{TRIPTYQUES_FOLDER}triptyques_chap1to5_sacr_chap.csv")


In [7]:
auto_chap1to5 = auto[auto["chapter"].isin([1, 2, 3, 4, 5])].copy()

print("Triptyques automatiques chap. 1 à 5 :", len(auto_chap1to5))
print("Triptyques manuels :", len(manuel))

key_cols = ["AlignID_sujet", "AlignID_verbe", "AlignID_objet"]

missing_auto = [c for c in key_cols if c not in auto_chap1to5.columns]
missing_manuel = [c for c in key_cols if c not in manuel.columns]

if missing_auto or missing_manuel:
    raise ValueError(f"Colonnes manquantes - auto: {missing_auto}, manuel: {missing_manuel}")

Triptyques automatiques chap. 1 à 5 : 1259
Triptyques manuels : 1255


In [8]:
for df in [auto_chap1to5, manuel]:
    df["cle_triptyque"] = (
        df[key_cols]
        .fillna("")
        .astype(str)
        .apply(lambda row: " || ".join(row), axis=1)
    )


In [9]:
cles_auto = set(auto_chap1to5["cle_triptyque"])
cles_manuel = set(manuel["cle_triptyque"])

communs = cles_auto & cles_manuel
seulement_auto = cles_auto - cles_manuel
seulement_manuel = cles_manuel - cles_auto

print("Triptyques communs :", len(communs))
print("Triptyques seulement automatiques :", len(seulement_auto))
print("Triptyques seulement manuels :", len(seulement_manuel))


Triptyques communs : 705
Triptyques seulement automatiques : 554
Triptyques seulement manuels : 548


In [10]:
df_communs = auto_chap1to5[auto_chap1to5["cle_triptyque"].isin(communs)].copy()

df_seulement_auto = auto_chap1to5[auto_chap1to5["cle_triptyque"].isin(seulement_auto)].copy()
df_seulement_auto["source_comparaison"] = "seulement_automatique"

df_seulement_manuel = manuel[manuel["cle_triptyque"].isin(seulement_manuel)].copy()
df_seulement_manuel["source_comparaison"] = "seulement_manuel"

differences_triptyques = pd.concat(
    [df_seulement_auto, df_seulement_manuel],
    ignore_index=True
)

print("Communs :", len(df_communs))
print("Différences :", len(differences_triptyques))

differences_triptyques[
    ["source_comparaison", "Phrase", "Sujet", "Verbe", "Objet",
     "AlignID_sujet", "AlignID_verbe", "AlignID_objet",
     "Dep_sujet", "Dep_verbe", "Dep_objet"]
].head(100)

Communs : 705
Différences : 1104


,source_comparaison,Phrase,Sujet,Verbe,Objet,AlignID_sujet,AlignID_verbe,AlignID_objet,Dep_sujet,Dep_verbe,Dep_objet
0,seulement_automatique,"Mais comment il avait fait fortune, c’ est ce ...",NaN,apprendre,l’,NaN,532,531.0,NaN,advcl,obj
1,seulement_automatique,"En tout cas, il n’ était prodigue de rien, mai...",il,apportait,En tout cas,565.0,567,536.0,nsubj,root,obl:mod
2,seulement_automatique,"En tout cas, il n’ était prodigue de rien, mai...",il,apportait,de rien,565.0,567,543.0,nsubj,root,obl:arg
3,seulement_automatique,C’ était un homme qui avait dû voyager partout...,qui,dû,NaN,729.0,731,NaN,nsubj,acl:relcl,NaN
4,seulement_automatique,Ceux qui avaient l’ honneur de le connaître un...,Ceux,attestaient,NaN,765.0,779,NaN,nsubj,root,NaN
...,...,...,...,...,...,...,...,...,...,...,...
95,seulement_automatique,« J’ ai vingt mille livres( 500 000 F) dé...,J’,ai,vingt mille livres 500 000 F,5725.0,5726,5729.0,nsubj,ccomp,obj
96,seulement_automatique,« J’ ai vingt mille livres( 500 000 F) dé...,NaN,déposées,chez Baring frères,NaN,5735,5737.0,NaN,acl,obl:arg
97,seulement_automatique,Je les risquerai volontiers…,Je,risquerai,les,5740.0,5742,5741.0,nsubj,root,obj
98,seulement_automatique,– Vingt mille livres ! s’ écria John Sulliv...,John Sullivan,écria,NaN,5751.0,5750,NaN,nsubj,parataxis,NaN


In [11]:
resume_dependances = (
    differences_triptyques
    .groupby(["source_comparaison", "Dep_sujet", "Dep_verbe", "Dep_objet"], dropna=False)
    .size()
    .reset_index(name="nombre")
    .sort_values("nombre", ascending=False)
)

resume_dependances


,source_comparaison,Dep_sujet,Dep_verbe,Dep_objet,nombre
123,seulement_manuel,nsubj,root,obl:mod,59
33,seulement_automatique,nsubj,root,obl:mod,56
31,seulement_automatique,nsubj,root,obj,54
121,seulement_manuel,nsubj,root,obj,52
34,seulement_automatique,nsubj,root,NaN,35
...,...,...,...,...,...
149,seulement_manuel,NaN,advcl,expl:comp,1
162,seulement_manuel,NaN,nsubj,NaN,1
156,seulement_manuel,NaN,conj,obl:arg,1
157,seulement_manuel,NaN,cop,NaN,1


In [ ]:
# differences_triptyques.to_csv("diff.csv", index=False)
# resume_dependances.to_csv("resume_dependances.csv", index=False)

# print("Fichiers créés : diff.csv et resume_dependances.csv")


Fichiers créés : diff.csv et resume_dependances.csv
